In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 4.4 MB/s eta 0:00:00


In [3]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score


In [4]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']
#Load Datasets
df = pd.read_csv(url, names=columns)

df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals]=df[cols_with_missing_vals].replace(0,np.nan)

#Imputing these nan values with mean of the respective column
df.fillna(df.mean(),inplace=True)

print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


In [7]:

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


# using bayesian optimizer

In [11]:
# Creating a study object and optimize the objective function
study=optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler()) # We aim to maximize accuracy
study.optimize(objective , n_trials=75)

[I 2026-06-22 07:56:54,011] A new study created in memory with name: no-name-4ad7a955-09cb-4a2d-a893-03e3469d1974
[I 2026-06-22 07:56:54,630] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 99, 'max_depth': 20}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:56:55,728] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 196, 'max_depth': 6}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:56:56,151] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 76, 'max_depth': 5}. Best is trial 2 with value: 0.7709497206703911.
[I 2026-06-22 07:56:56,749] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 100, 'max_depth': 8}. Best is trial 2 with value: 0.7709497206703911.
[I 2026-06-22 07:56:57,756] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 170, 'max_depth': 13}. Best is trial 2 with value: 0.770949720

In [12]:
print(f"Best Trail accuracy : {study.best_trial.values}")
print(f"Best hyperparameter : {study.best_trial.params}")

Best Trail accuracy : [0.7802607076350093]
Best hyperparameter : {'n_estimators': 123, 'max_depth': 17}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# Using Random Sample

In [16]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=75)  # Run 50 trials to find the best hyperparameters

[I 2026-06-22 07:16:47,965] A new study created in memory with name: no-name-40bab72f-532b-4267-94f0-58c50379e8e2
[I 2026-06-22 07:16:48,791] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 85, 'max_depth': 19}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:16:49,568] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 83, 'max_depth': 14}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:16:51,274] Trial 2 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 173, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:16:52,243] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 104, 'max_depth': 11}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:16:53,519] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 156, 'max_depth': 18}. Best is trial 4 with value: 0.7728119

In [17]:
print(f"Best Trail accuracy : {study.best_trial.values}")
print(f"Best hyperparameter : {study.best_trial.params}")

Best Trail accuracy : [0.7783985102420856]
Best hyperparameter : {'n_estimators': 80, 'max_depth': 12}


In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.76


#Using GridSearch  

In [19]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [20]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-06-22 07:19:49,715] A new study created in memory with name: no-name-dbfcbe87-db29-4148-90c8-818ad702527c
[I 2026-06-22 07:19:50,423] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:19:51,833] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-06-22 07:19:52,295] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-22 07:19:53,224] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-06-22 07:19:54,136] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [21]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [22]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


# Optuna Visualization

In [23]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [33]:
# 1. Optimization History
plot_optimization_history(study).show()

In [34]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [35]:
# 3. Slice Plot
plot_slice(study).show()

In [36]:
# 4. Contour Plot
plot_contour(study).show()

In [37]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

# Define By Run (Dynamic Search Space)

In [14]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [15]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [16]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-22 08:03:04,638] A new study created in memory with name: no-name-0fe19fa9-d28f-49f9-b62f-2720ad72f0a0
[I 2026-06-22 08:03:04,703] Trial 0 finished with value: 0.7709497206703911 and parameters: {'classifier': 'SVM', 'C': 1.4883856983490695, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-22 08:03:07,738] Trial 1 finished with value: 0.7653631284916201 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 178, 'learning_rate': 0.031055377514805766, 'max_depth': 14, 'min_samples_split': 10, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-06-22 08:03:07,765] Trial 2 finished with value: 0.7895716945996275 and parameters: {'classifier': 'SVM', 'C': 0.14204820837153184, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 2 with value: 0.7895716945996275.
[I 2026-06-22 08:03:09,305] Trial 3 finished with value: 0.756052141527002 and parameters: {'classifier': 'RandomForest', 'n_estimators'

In [17]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.14204820837153184, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [18]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.770950,2026-06-22 08:03:04.642155,2026-06-22 08:03:04.703637,0 days 00:00:00.061482,1.488386,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.765363,2026-06-22 08:03:04.705560,2026-06-22 08:03:07.738416,0 days 00:00:03.032856,NaN,NaN,GradientBoosting,NaN,NaN,0.031055,14.0,8.0,10.0,178.0,COMPLETE
2,2,0.789572,2026-06-22 08:03:07.739450,2026-06-22 08:03:07.765133,0 days 00:00:00.025683,0.142048,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.756052,2026-06-22 08:03:07.766114,2026-06-22 08:03:09.305776,0 days 00:00:01.539662,NaN,True,RandomForest,NaN,NaN,NaN,16.0,6.0,3.0,282.0,COMPLETE
4,4,0.769088,2026-06-22 08:03:09.307045,2026-06-22 08:03:09.347366,0 days 00:00:00.040321,0.267583,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.767225,2026-06-22 08:04:01.839957,2026-06-22 08:04:02.608043,0 days 00:00:00.768086,NaN,False,RandomForest,NaN,NaN,NaN,12.0,4.0,7.0,146.0,COMPLETE
96,96,0.716946,2026-06-22 08:04:02.609126,2026-06-22 08:04:02.643737,0 days 00:00:00.034611,0.221053,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2026-06-22 08:04:02.644754,2026-06-22 08:04:02.675615,0 days 00:00:00.030861,0.115754,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-06-22 08:04:02.676607,2026-06-22 08:04:02.708803,0 days 00:00:00.032196,0.101323,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [19]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,79
GradientBoosting,11
RandomForest,10


In [20]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.749450
RandomForest,0.766108
SVM,0.777880
